# Cartes par quart de département

Génère, pour chaque département choisi, 4 cartes (un quart NO/NE/SO/SE de son emprise) superposant les isochrones Géofer et la densité de population des carreaux INSEE 200x200 m, avec les zones non desservies mises en évidence — même logique que `app.py`, réutilisée telle quelle.

Chaque quart est exporté en PNG (impression/aperçu rapide) et en HTML interactif, dans `Output/`, puis envoyé vers le dataset [antoinechevre/Analyse_gare](https://huggingface.co/datasets/antoinechevre/Analyse_gare) (sous `cartes/`), en best-effort — nécessite un token HF avec droit d'écriture (`HF_TOKEN` en variable d'environnement, ou déjà connecté via `huggingface-cli login`).

In [ ]:
import os
import sys

import contextily as cx
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
from huggingface_hub import HfApi
from matplotlib.colors import LinearSegmentedColormap
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.validation import make_valid

sys.path.insert(0, ".")
import app

OUTPUT_DIR = "Output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CARREAUX_CMAP = LinearSegmentedColormap.from_list("carreaux", app.CARREAUX_COLOR_SCALE)

# Codes INSEE des départements à traiter. Pour un export national complet :
# DEPARTEMENTS = app.load_departements()["code"].tolist()
DEPARTEMENTS = ["51", "08", "02"]  # Marne, Ardennes, Aisne

# Cartes envoyées vers ce dataset HF après génération (cf. envoyer_carte_vers_hf
# plus bas) — nécessite un token avec droit d'écriture (HF_TOKEN en variable
# d'environnement, ou déjà connecté via `huggingface-cli login`).
HF_DATASET_CARTES = "antoinechevre/Analyse_gare"


def envoyer_carte_vers_hf(chemin_local, nom_fichier_hf):
    """Best-effort, comme hf_cache.envoyer_vers_hf dans les apps sœurs :
    n'interrompt jamais la génération des cartes en cas d'échec (token
    absent/sans droit d'écriture, réseau...)."""
    try:
        HfApi().upload_file(
            path_or_fileobj=chemin_local,
            path_in_repo=nom_fichier_hf,
            repo_id=HF_DATASET_CARTES,
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )
        return True
    except Exception as e:
        print(f"[hf] échec envoi {nom_fichier_hf} : {type(e).__name__}: {e}")
        return False

In [ ]:
def decouper_en_quadrants(dept_geom):
    """Découpe l'emprise d'un département en 4 quarts (NO/NE/SO/SE), clippés à sa forme réelle."""
    minx, miny, maxx, maxy = dept_geom.bounds
    midx, midy = (minx + maxx) / 2, (miny + maxy) / 2
    boites = {
        "NO": box(minx, midy, midx, maxy),
        "NE": box(midx, midy, maxx, maxy),
        "SO": box(minx, miny, midx, midy),
        "SE": box(midx, miny, maxx, midy),
    }
    quadrants = {}
    for nom, boite in boites.items():
        geom = dept_geom.intersection(boite)
        if not geom.is_empty:
            quadrants[nom] = geom
    return quadrants

In [ ]:
def donnees_quadrant(quadrant_geom, quadrant_key, gares, insee_path, offre, frequentation):
    """Gares, isochrones, carreaux INSEE (avec statut desservi/non desservi), offre 2026 et
    fréquentation pour un quart de département.

    quadrant_key doit être unique par quart (ex. "51_NO") : app.load_insee_carreaux est mise en cache
    par Streamlit sur cet identifiant, pas sur la géométrie elle-même.
    """
    bounds = quadrant_geom.bounds
    gares_quad = gares[
        gares["wgs84Lon"].between(bounds[0], bounds[2]) & gares["wgs84Lat"].between(bounds[1], bounds[3])
    ]
    codes_quad = set(gares_quad["codeUic"])

    isochrones_quad = {}
    for mode, (path, _) in app.ISOCHRONE_FILES.items():
        full = app.load_isochrones(path)
        isochrones_quad[mode] = full[full["code_uic"].isin(codes_quad)]

    carreaux = app.load_insee_carreaux(insee_path, quadrant_geom, quadrant_key)
    if not carreaux.empty:
        # make_valid avant union : des isochrones Géofer peuvent être
        # topologiquement invalides (auto-intersections), ce qui fait planter
        # GEOS sur unary_union sinon (cf. app.py, main(), même précaution).
        served_geoms = [make_valid(geom) for gdf in isochrones_quad.values() for geom in gdf.geometry]
        union_geom = unary_union(served_geoms) if served_geoms else None
        carreaux["desservi"] = carreaux.intersects(union_geom) if union_geom is not None else False

    # Même jointure que app.py (main()) : gares du quart x offre/fréquentation par codeUic.
    base_gares = gares_quad[["codeUic", "wgs84Lat", "wgs84Lon"]]
    offre_quad = base_gares.merge(offre, on="codeUic", how="inner")
    frequentation_quad = base_gares.merge(frequentation, on="codeUic", how="inner")

    return gares_quad, isochrones_quad, carreaux, offre_quad, frequentation_quad

In [ ]:
def rendre_png(
    quadrant_geom, gares_quad, isochrones_quad, carreaux, offre_quad, frequentation_quad,
    titre, chemin_sortie, color_field="pop",
):
    fig, ax = plt.subplots(figsize=(10, 10))

    if not carreaux.empty:
        carreaux_3857 = carreaux.to_crs(3857)
        desservi = carreaux_3857[carreaux_3857["desservi"]]
        non_desservi = carreaux_3857[~carreaux_3857["desservi"]]
        if not desservi.empty:
            desservi.plot(ax=ax, color="#c8ced6", alpha=0.35, linewidth=0)
        if not non_desservi.empty:
            non_desservi.plot(
                ax=ax, column=color_field, cmap=CARREAUX_CMAP, alpha=0.92, linewidth=0,
                legend=True, legend_kwds={"label": color_field, "shrink": 0.6},
            )

    for mode, gdf in isochrones_quad.items():
        if gdf.empty:
            continue
        _, couleur = app.ISOCHRONE_FILES[mode]
        gdf.to_crs(3857).plot(ax=ax, facecolor=couleur, edgecolor=couleur, alpha=0.3, linewidth=1.5)

    if not gares_quad.empty:
        gares_pts = gpd.GeoDataFrame(
            gares_quad,
            geometry=gpd.points_from_xy(gares_quad["wgs84Lon"], gares_quad["wgs84Lat"]),
            crs="EPSG:4326",
        ).to_crs(3857)
        gares_pts.plot(ax=ax, color="#581012", marker="^", markersize=40, zorder=5)

    # Offre 2026 : un aplat matplotlib ne rend pas bien un camembert par point
    # (contrairement au DivIcon HTML de app.py) — repli sur un point coloré
    # selon la catégorie dominante (TER/Intercités/TGV), taille ~ sqrt(total),
    # même principe de proportionnalité en aire que app.py.
    if not offre_quad.empty:
        offre_pts = gpd.GeoDataFrame(
            offre_quad,
            geometry=gpd.points_from_xy(offre_quad["wgs84Lon"], offre_quad["wgs84Lat"]),
            crs="EPSG:4326",
        ).to_crs(3857)
        colonnes_categorie = list(app.OFFRE_CATEGORIES)
        categorie_dominante = offre_pts[colonnes_categorie].idxmax(axis=1)
        couleurs = categorie_dominante.map({c: app.OFFRE_CATEGORIES[c][1] for c in colonnes_categorie})
        taille = 30 + 200 * (offre_pts["totalClasse"] / offre_pts["totalClasse"].max()) ** 0.5
        offre_pts.plot(ax=ax, color=couleurs, markersize=taille, alpha=0.85, zorder=6, edgecolor="black", linewidth=0.5)

    if not frequentation_quad.empty:
        freq_pts = gpd.GeoDataFrame(
            frequentation_quad,
            geometry=gpd.points_from_xy(frequentation_quad["wgs84Lon"], frequentation_quad["wgs84Lat"]),
            crs="EPSG:4326",
        ).to_crs(3857)
        taille = 30 + 300 * (freq_pts["voyageurs"] / freq_pts["voyageurs"].max()) ** 0.5
        freq_pts.plot(
            ax=ax, color=app.FREQUENTATION_COLOR, markersize=taille, alpha=0.6, zorder=4,
            edgecolor="black", linewidth=0.5,
        )

    quadrant_3857 = gpd.GeoSeries([quadrant_geom], crs="EPSG:4326").to_crs(3857).iloc[0]
    minx, miny, maxx, maxy = quadrant_3857.bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    # contextily ne gère pas le multiplexage de sous-domaines {s} (convention
    # Leaflet) : sous-domaine "a" fixé plutôt que app.carto_tile_url() brut.
    # attribution en texte brut (pas app.CARTO_ATTR, qui est du HTML — rendu
    # comme du texte littéral sur un export matplotlib statique).
    cx.add_basemap(
        ax, source=app.carto_tile_url("light_all").replace("{s}", "a"),
        attribution="© OpenStreetMap contributors, © CARTO",
    )
    ax.set_axis_off()
    ax.set_title(titre, fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(chemin_sortie, dpi=150)
    plt.close(fig)

In [ ]:
def rendre_html(
    quadrant_geom, gares_quad, isochrones_quad, carreaux, offre_quad, frequentation_quad,
    color_field, color_label, chemin_sortie,
):
    bounds = quadrant_geom.bounds
    centre = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
    m = folium.Map(
        location=centre, tiles=app.carto_tile_url("light_all"), attr=app.CARTO_ATTR, prefer_canvas=True,
    )

    if not carreaux.empty:
        unserved = carreaux[~carreaux["desservi"]]
        source_echelle = unserved if not unserved.empty else carreaux
        colormap = folium.LinearColormap(
            colors=app.CARREAUX_COLOR_SCALE,
            vmin=float(source_echelle[color_field].min()),
            vmax=float(source_echelle[color_field].max()),
            caption=f"{color_label} — carreaux non desservis",
        )

        def style_carreau(feature, cf=color_field, cm=colormap):
            if feature["properties"]["desservi"]:
                return {"fillColor": "#c8ced6", "color": "#9aa3af", "weight": 0, "fillOpacity": 0.35}
            return {"fillColor": cm(feature["properties"][cf]), "color": "#581012", "weight": 0, "fillOpacity": 0.92}

        folium.GeoJson(
            carreaux,
            style_function=style_carreau,
            tooltip=folium.GeoJsonTooltip(
                fields=["pop", "niveau_vie", "taux_pauvrete", "part_65p", "desservi"],
                aliases=["Population", "Revenu moyen (€)", "Taux de pauvreté (%)", "Part 65 ans+ (%)", "Desservi"],
                localize=True,
            ),
        ).add_to(m)
        colormap.add_to(m)

    for mode, gdf in isochrones_quad.items():
        if gdf.empty:
            continue
        _, couleur = app.ISOCHRONE_FILES[mode]
        folium.GeoJson(
            gdf,
            name=mode,
            style_function=lambda f, c=couleur: {"color": c, "weight": 2, "fill": True, "fillColor": c, "fillOpacity": 0.3},
        ).add_to(m)

    # Offre 2026 et fréquentation : mêmes fonctions de rendu que app.py
    # (build_map), échelle de taille locale au quart affiché (même principe
    # que app.py — un max national écraserait la variation locale).
    if not offre_quad.empty:
        offre_layer = folium.FeatureGroup(name="Offre 2026 (TER / Intercités / TGV)")
        offre_max = offre_quad["totalClasse"].max()
        for _, gare_offre in offre_quad.iterrows():
            rayon = app.OFFRE_MIN_RADIUS_PX + (app.OFFRE_MAX_RADIUS_PX - app.OFFRE_MIN_RADIUS_PX) * (
                gare_offre["totalClasse"] / offre_max
            ) ** 0.5
            folium.Marker(
                [gare_offre["wgs84Lat"], gare_offre["wgs84Lon"]],
                icon=folium.DivIcon(
                    html=app.offre_pie_svg(gare_offre, rayon),
                    icon_size=(rayon * 2, rayon * 2),
                    icon_anchor=(rayon, rayon),
                ),
                tooltip=f"{gare_offre['nomGare']} — {int(gare_offre['totalClasse'])} trains/jour",
                popup=folium.Popup(app.offre_popup(gare_offre), max_width=220),
            ).add_to(offre_layer)
        offre_layer.add_to(m)
        m.get_root().html.add_child(folium.Element(app.script_legende_offre()))

    if not frequentation_quad.empty:
        frequentation_layer = folium.FeatureGroup(name=f"Fréquentation {app.FREQUENTATION_ANNEE} (voyageurs/an)")
        frequentation_max = frequentation_quad["voyageurs"].max()
        for _, gare_freq in frequentation_quad.iterrows():
            rayon = app.FREQUENTATION_MIN_RADIUS_PX + (
                app.FREQUENTATION_MAX_RADIUS_PX - app.FREQUENTATION_MIN_RADIUS_PX
            ) * (gare_freq["voyageurs"] / frequentation_max) ** 0.5
            folium.Marker(
                [gare_freq["wgs84Lat"], gare_freq["wgs84Lon"]],
                icon=folium.DivIcon(
                    html=app.frequentation_bubble_svg(rayon),
                    icon_size=(rayon * 2, rayon * 2),
                    icon_anchor=(rayon, rayon),
                ),
                tooltip=(
                    f"{gare_freq['nomGare']} — {int(gare_freq['voyageurs']):,} voyageurs/an "
                    f"({app.FREQUENTATION_ANNEE})".replace(",", " ")
                ),
            ).add_to(frequentation_layer)
        frequentation_layer.add_to(m)

    for _, gare in gares_quad.iterrows():
        folium.Marker(
            [gare["wgs84Lat"], gare["wgs84Lon"]],
            tooltip=gare["nomGare"],
            icon=folium.Icon(color="darkred", icon="train", prefix="fa"),
        ).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    m.save(chemin_sortie)

In [ ]:
gares = app.load_gares()
departements = app.load_departements()
insee_path = app.get_insee_local_path()
offre = app.load_offre_2026()
frequentation = app.load_frequentation()

for dept_code in DEPARTEMENTS:
    dept = departements[departements["code"] == dept_code].iloc[0]
    quadrants = decouper_en_quadrants(dept.geometry)

    for nom_quad, quad_geom in quadrants.items():
        quadrant_key = f"{dept_code}_{nom_quad}"
        gares_quad, isochrones_quad, carreaux, offre_quad, frequentation_quad = donnees_quadrant(
            quad_geom, quadrant_key, gares, insee_path, offre, frequentation
        )

        titre = f"{dept['nom']} ({dept_code}) — {nom_quad}"
        base_nom = f"{dept_code}_{nom_quad}"
        chemin_png = f"{OUTPUT_DIR}/{base_nom}.png"
        chemin_html = f"{OUTPUT_DIR}/{base_nom}.html"

        rendre_png(quad_geom, gares_quad, isochrones_quad, carreaux, offre_quad, frequentation_quad, titre, chemin_png)
        rendre_html(
            quad_geom, gares_quad, isochrones_quad, carreaux, offre_quad, frequentation_quad,
            "pop", "Population", chemin_html,
        )

        envoi_png = envoyer_carte_vers_hf(chemin_png, f"cartes/{base_nom}.png")
        envoi_html = envoyer_carte_vers_hf(chemin_html, f"cartes/{base_nom}.html")
        statut_hf = "✓ envoyé" if envoi_png and envoi_html else "⚠ échec envoi HF"

        print(f"{base_nom} : {len(carreaux)} carreaux, {len(gares_quad)} gares — {statut_hf}")

print("Terminé —", OUTPUT_DIR, "et", HF_DATASET_CARTES)